# Semana 9 — Exercícios opcionais de SQL com dados limpos

## Contextos usados

Esta lista usa três conjuntos de dados fictícios, pequenos e já limpos:

1. **Cursos online** — alunos, cursos, matrículas e aulas assistidas.
2. **Fiscal** — notas fiscais, itens de notas, clientes e produtos.
3. **Financeiro** — contas a pagar, contas a receber e movimentações bancárias.

O objetivo é praticar apenas SQL, sem precisar fazer limpeza de dados.

### Conteúdos praticados

- `SELECT`, `WHERE`, `ORDER BY`
- `INNER JOIN` e `LEFT JOIN`
- `GROUP BY`, `SUM`, `COUNT`, `AVG`
- `HAVING`
- `DATE_TRUNC` e `EXTRACT`
- Perguntas de negócio usando SQL


## 1. Preparação do ambiente

Execute a célula abaixo antes de começar. Ela cria os dados em memória e registra as tabelas no DuckDB.


In [2]:
# Caso esteja no Google Colab e precise instalar o DuckDB, descomente a linha abaixo:
# !pip install duckdb pandas

import pandas as pd
import duckdb

con = duckdb.connect()

# -----------------------------
# DADOS: CURSOS ONLINE
# -----------------------------

alunos = pd.DataFrame({
    'aluno_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'nome_aluno': ['Ana', 'Bruno', 'Carla', 'Diego', 'Elisa', 'Felipe', 'Giovana', 'Hugo'],
    'estado': ['SP', 'SC', 'PR', 'SP', 'RJ', 'SC', 'MG', 'SP'],
    'data_cadastro': pd.to_datetime([
        '2024-01-10', '2024-01-15', '2024-02-04', '2024-02-20',
        '2024-03-01', '2024-03-18', '2024-04-05', '2024-04-22'
    ])
})

cursos = pd.DataFrame({
    'curso_id': [101, 102, 103, 104, 105],
    'nome_curso': ['SQL para BI', 'Python Básico', 'Power BI', 'Excel Avançado', 'Data Warehouse'],
    'categoria': ['Dados', 'Programação', 'BI', 'Produtividade', 'Dados'],
    'carga_horaria': [20, 30, 25, 18, 35],
    'preco': [199.90, 249.90, 299.90, 149.90, 399.90]
})

matriculas = pd.DataFrame({
    'matricula_id': [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012],
    'aluno_id': [1,1,2,3,3,4,5,6,6,7,8,8],
    'curso_id': [101,103,101,102,105,104,103,101,105,102,101,104],
    'data_matricula': pd.to_datetime([
        '2024-02-01','2024-02-10','2024-02-12','2024-03-05','2024-03-10','2024-03-15',
        '2024-04-01','2024-04-12','2024-04-20','2024-05-02','2024-05-09','2024-05-18'
    ]),
    'status': ['Concluído','Em andamento','Concluído','Cancelado','Em andamento','Concluído',
               'Em andamento','Concluído','Em andamento','Cancelado','Concluído','Em andamento']
})

aulas_assistidas = pd.DataFrame({
    'registro_id': [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
    'matricula_id': [1001,1001,1002,1003,1004,1005,1006,1007,1008,1008,1009,1010,1011,1011,1012],
    'data_aula': pd.to_datetime([
        '2024-02-03','2024-02-08','2024-02-12','2024-02-14','2024-03-07','2024-03-12',
        '2024-03-17','2024-04-05','2024-04-14','2024-04-21','2024-04-23','2024-05-04',
        '2024-05-11','2024-05-15','2024-05-20'
    ]),
    'minutos_assistidos': [45, 60, 50, 80, 20, 110, 75, 40, 65, 70, 95, 25, 60, 55, 35]
})

# -----------------------------
# DADOS: FISCAL
# -----------------------------

clientes_fiscais = pd.DataFrame({
    'cliente_id': [1,2,3,4,5,6],
    'razao_social': ['Alpha Ltda', 'Beta Comércio', 'Casa Norte', 'Delta Serviços', 'Escola Saber', 'Farmácia Vida'],
    'uf': ['SP','SC','PR','SP','RJ','MG'],
    'segmento': ['Indústria','Comércio','Comércio','Serviços','Educação','Saúde']
})

produtos_fiscais = pd.DataFrame({
    'produto_id': [201,202,203,204,205],
    'descricao_produto': ['Notebook', 'Monitor', 'Licença Software', 'Consultoria', 'Treinamento'],
    'ncm': ['84713012','85285220','00000000','00000000','00000000'],
    'tipo': ['Produto','Produto','Serviço','Serviço','Serviço']
})

notas_fiscais = pd.DataFrame({
    'nota_id': [5001,5002,5003,5004,5005,5006,5007,5008],
    'cliente_id': [1,2,3,4,1,5,6,2],
    'data_emissao': pd.to_datetime(['2024-01-08','2024-01-20','2024-02-10','2024-02-18','2024-03-05','2024-03-22','2024-04-09','2024-04-25']),
    'status_nota': ['Autorizada','Autorizada','Cancelada','Autorizada','Autorizada','Autorizada','Autorizada','Cancelada'],
    'uf_destino': ['SP','SC','PR','SP','SP','RJ','MG','SC']
})

itens_notas = pd.DataFrame({
    'item_id': [1,2,3,4,5,6,7,8,9,10,11],
    'nota_id': [5001,5001,5002,5003,5004,5004,5005,5006,5007,5008,5008],
    'produto_id': [201,203,202,201,204,205,203,205,202,201,204],
    'quantidade': [2,1,3,1,1,2,4,3,2,1,1],
    'valor_unitario': [3500.00,800.00,1200.00,3600.00,2500.00,900.00,750.00,850.00,1150.00,3400.00,2200.00],
    'aliquota_imposto': [0.12,0.05,0.12,0.12,0.05,0.05,0.05,0.05,0.12,0.12,0.05]
})

itens_notas['valor_item'] = itens_notas['quantidade'] * itens_notas['valor_unitario']
itens_notas['valor_imposto'] = itens_notas['valor_item'] * itens_notas['aliquota_imposto']

# -----------------------------
# DADOS: FINANCEIRO
# -----------------------------

centros_custo = pd.DataFrame({
    'centro_custo_id': [1,2,3,4],
    'centro_custo': ['Administrativo', 'Comercial', 'Tecnologia', 'Operações']
})

contas_pagar = pd.DataFrame({
    'conta_pagar_id': [9001,9002,9003,9004,9005,9006,9007,9008],
    'fornecedor': ['Energia Sul', 'Aluguel Central', 'CloudData', 'Agência Ads', 'Internet Fibra', 'Manutenção Pro', 'SoftPlan', 'Transporte Rápido'],
    'centro_custo_id': [1,1,3,2,3,4,3,4],
    'data_vencimento': pd.to_datetime(['2024-01-10','2024-01-15','2024-02-05','2024-02-20','2024-03-10','2024-03-25','2024-04-10','2024-04-18']),
    'data_pagamento': pd.to_datetime(['2024-01-09','2024-01-18','2024-02-05','2024-02-22','2024-03-09',None,'2024-04-12',None]),
    'valor': [850.00, 3200.00, 1800.00, 2500.00, 430.00, 1250.00, 2100.00, 980.00],
    'status': ['Pago','Pago','Pago','Pago','Pago','Aberto','Pago','Aberto']
})

contas_receber = pd.DataFrame({
    'conta_receber_id': [8001,8002,8003,8004,8005,8006,8007,8008],
    'cliente': ['Alpha Ltda', 'Beta Comércio', 'Casa Norte', 'Delta Serviços', 'Escola Saber', 'Farmácia Vida', 'Alpha Ltda', 'Beta Comércio'],
    'data_vencimento': pd.to_datetime(['2024-01-12','2024-01-28','2024-02-15','2024-02-28','2024-03-12','2024-03-30','2024-04-12','2024-04-28']),
    'data_recebimento': pd.to_datetime(['2024-01-12','2024-01-30',None,'2024-02-28','2024-03-15',None,'2024-04-12','2024-04-30']),
    'valor': [7800.00, 3600.00, 4200.00, 2500.00, 5100.00, 2300.00, 6900.00, 3100.00],
    'status': ['Recebido','Recebido','Aberto','Recebido','Recebido','Aberto','Recebido','Recebido']
})

movimentacoes_bancarias = pd.DataFrame({
    'movimento_id': [1,2,3,4,5,6,7,8,9,10],
    'data_movimento': pd.to_datetime(['2024-01-09','2024-01-12','2024-01-18','2024-01-30','2024-02-05','2024-02-22','2024-02-28','2024-03-09','2024-03-15','2024-04-12']),
    'tipo_movimento': ['Saída','Entrada','Saída','Entrada','Saída','Saída','Entrada','Saída','Entrada','Entrada'],
    'categoria': ['Energia','Recebimento Cliente','Aluguel','Recebimento Cliente','Cloud','Marketing','Recebimento Cliente','Internet','Recebimento Cliente','Recebimento Cliente'],
    'valor': [850.00, 7800.00, 3200.00, 3600.00, 1800.00, 2500.00, 2500.00, 430.00, 5100.00, 6900.00]
})

# Registro das tabelas no DuckDB
for nome, df in {
    'alunos': alunos,
    'cursos': cursos,
    'matriculas': matriculas,
    'aulas_assistidas': aulas_assistidas,
    'clientes_fiscais': clientes_fiscais,
    'produtos_fiscais': produtos_fiscais,
    'notas_fiscais': notas_fiscais,
    'itens_notas': itens_notas,
    'centros_custo': centros_custo,
    'contas_pagar': contas_pagar,
    'contas_receber': contas_receber,
    'movimentacoes_bancarias': movimentacoes_bancarias
}.items():
    con.register(nome, df)

print('Tabelas criadas com sucesso!')


Tabelas criadas com sucesso!


## 2. Conferindo as tabelas disponíveis

Execute as consultas abaixo para visualizar uma amostra dos dados.


In [3]:
con.execute('SHOW TABLES').df()

,name
0,alunos
1,aulas_assistidas
2,centros_custo
3,clientes_fiscais
4,contas_pagar
5,contas_receber
6,cursos
7,itens_notas
8,matriculas
9,movimentacoes_bancarias


In [4]:
con.execute('SELECT * FROM cursos LIMIT 5').df()

,curso_id,nome_curso,categoria,carga_horaria,preco
0,101,SQL para BI,Dados,20,199.9
1,102,Python Básico,Programação,30,249.9
2,103,Power BI,BI,25,299.9
3,104,Excel Avançado,Produtividade,18,149.9
4,105,Data Warehouse,Dados,35,399.9


In [5]:
con.execute('SELECT * FROM notas_fiscais LIMIT 5').df()

,nota_id,cliente_id,data_emissao,status_nota,uf_destino
0,5001,1,2024-01-08,Autorizada,SP
1,5002,2,2024-01-20,Autorizada,SC
2,5003,3,2024-02-10,Cancelada,PR
3,5004,4,2024-02-18,Autorizada,SP
4,5005,1,2024-03-05,Autorizada,SP


In [6]:
con.execute('SELECT * FROM contas_pagar LIMIT 5').df()

,conta_pagar_id,fornecedor,centro_custo_id,data_vencimento,data_pagamento,valor,status
0,9001,Energia Sul,1,2024-01-10,2024-01-09,850.0,Pago
1,9002,Aluguel Central,1,2024-01-15,2024-01-18,3200.0,Pago
2,9003,CloudData,3,2024-02-05,2024-02-05,1800.0,Pago
3,9004,Agência Ads,2,2024-02-20,2024-02-22,2500.0,Pago
4,9005,Internet Fibra,3,2024-03-10,2024-03-09,430.0,Pago


# Parte A — Cursos online

Nesta parte, você irá responder perguntas sobre alunos, cursos, matrículas e aulas assistidas.


## Exercício 1 — Cursos mais caros

Liste todos os cursos com preço acima de R$ 200, mostrando:

- `nome_curso`
- `categoria`
- `carga_horaria`
- `preco`

Ordene do maior preço para o menor.


In [11]:
# Escreva sua consulta SQL aqui
query = """
SELECT
    nome_curso,
    categoria,
    carga_horaria,
    preco
FROM cursos
WHERE preco > 200
ORDER BY preco DESC;
"""
con.execute(query).df()

,nome_curso,categoria,carga_horaria,preco
0,Data Warehouse,Dados,35,399.9
1,Power BI,BI,25,299.9
2,Python Básico,Programação,30,249.9


## Exercício 2 — Matrículas por status

Conte quantas matrículas existem em cada status:

- `Concluído`
- `Em andamento`
- `Cancelado`

Ordene do maior volume para o menor.


In [28]:
query = """
SELECT
    COUNT (matricula_id),
    status
FROM matriculas
GROUP BY status
"""

con.execute(query).df()

,count(matricula_id),status
0,2,Cancelado
1,5,Concluído
2,5,Em andamento


## Exercício 3 — Alunos e cursos matriculados

Use `INNER JOIN` para listar as matrículas mostrando:

- nome do aluno
- estado do aluno
- nome do curso
- categoria do curso
- status da matrícula

Ordene pelo nome do aluno.


In [34]:
query = """
SELECT
    a.nome_aluno AS Nome,
    a.estado AS Estado,
    c.nome_curso AS Curso,
    c.categoria AS Categoria,
    m.status AS Status_Matricula
FROM matriculas m
INNER JOIN alunos a
    ON m.aluno_id = m.aluno_id
INNER JOIN cursos c
    ON m.curso_id = c.curso_id
ORDER BY Nome;   
"""

con.execute(query).df()

,Nome,Estado,Curso,Categoria,Status_Matricula
0,Ana,SP,SQL para BI,Dados,Concluído
1,Ana,SP,Python Básico,Programação,Cancelado
2,Ana,SP,Power BI,BI,Em andamento
3,Ana,SP,Excel Avançado,Produtividade,Em andamento
4,Ana,SP,Data Warehouse,Dados,Em andamento
...,...,...,...,...,...
91,Hugo,SP,Power BI,BI,Em andamento
92,Hugo,SP,Excel Avançado,Produtividade,Concluído
93,Hugo,SP,Data Warehouse,Dados,Em andamento
94,Hugo,SP,SQL para BI,Dados,Concluído


## Exercício 4 — Receita por categoria de curso

Calcule a receita potencial por categoria, considerando o preço do curso em cada matrícula.

Mostre:

- categoria
- quantidade de matrículas
- receita total
- preço médio dos cursos matriculados

Ordene pela maior receita total.


In [39]:
query = """
SELECT
    c.categoria AS categoria,
    COUNT(m.matricula_id) AS qtd_matriculas,
    SUM(c.preco) AS receita_total,
    AVG(preco) AS preco_medio
FROM matriculas m
INNER JOIN cursos c
    ON m.curso_id = c.curso_id
GROUP BY categoria
ORDER BY receita_total DESC;
"""

con.execute(query).df()

,categoria,qtd_matriculas,receita_total,preco_medio
0,Dados,6,1599.4,266.566667
1,BI,2,599.8,299.900000
2,Programação,2,499.8,249.900000
3,Produtividade,2,299.8,149.900000


## Exercício 5 — Categorias com receita acima de R$ 500

Usando `GROUP BY` e `HAVING`, liste apenas as categorias de curso com receita total acima de R$ 500.

Mostre:

- categoria
- receita total

Ordene pela receita total em ordem decrescente.


In [42]:
query = """
SELECT
    c.categoria AS categoria,
    SUM(c.preco) AS receita_total,
FROM matriculas m
INNER JOIN cursos c
    ON m.curso_id = c.curso_id
GROUP BY categoria
HAVING receita_total > 500
ORDER BY receita_total DESC;
"""

con.execute(query).df()

,categoria,receita_total
0,Dados,1599.4
1,BI,599.8


## Exercício 6 — Minutos assistidos por aluno

Calcule o total de minutos assistidos por aluno.

Use as tabelas:

- `alunos`
- `matriculas`
- `aulas_assistidas`

Mostre:

- nome do aluno
- estado
- total de minutos assistidos

Ordene do maior total para o menor.


In [48]:
query = """
SELECT
    a.nome_aluno AS Nome,
    a.estado AS estado,
    SUM(aa.minutos_assistidos) AS total_minutos
FROM matriculas m
INNER JOIN alunos a
    ON m.aluno_id = a.aluno_id
INNER JOIN aulas_assistidas aa
    ON m.matricula_id = aa.matricula_id
GROUP BY m.aluno_id, nome, estado
ORDER BY total_minutos DESC;    
"""

con.execute(query).df()

,Nome,estado,total_minutos
0,Felipe,SC,230.0
1,Ana,SP,155.0
2,Hugo,SP,150.0
3,Carla,PR,130.0
4,Bruno,SC,80.0
5,Diego,SP,75.0
6,Elisa,RJ,40.0
7,Giovana,MG,25.0


## Exercício 7 — Matrículas por mês

Use `DATE_TRUNC` para calcular a quantidade de matrículas por mês.

Mostre:

- mês de referência
- quantidade de matrículas

Ordene o resultado em ordem cronológica.


In [52]:
query = """
SELECT
    DATE_TRUNC ('month', CAST(data_matricula AS DATE)) as mes,
    COUNT(matricula_id) AS qtde_matriculas
FROM matriculas
GROUP BY mes
ORDER BY mes ASC;
"""

con.execute(query).df()

,mes,qtde_matriculas
0,2024-02-01,3
1,2024-03-01,3
2,2024-04-01,3
3,2024-05-01,3


In [53]:
print(con.execute('SELECT * FROM matriculas LIMIT 1').df())
# print(con.execute('SELECT * FROM alunos LIMIT 1').df())
# print(con.execute('SELECT * FROM aulas_assistidas LIMIT 1').df())
# alunos - aulas_assistidas - centros_custo - clientes_fiscais - contas_pagar - contas_receber
# cursos - itens_notas - matriculas - movimentacoes_bancarias - notas_fiscais - produtos_fiscais


   matricula_id  aluno_id  curso_id data_matricula     status
0          1001         1       101     2024-02-01  Concluído


# Parte B — Fiscal

Nesta parte, você irá responder perguntas sobre notas fiscais, produtos, clientes e impostos.


## Exercício 8 — Notas fiscais autorizadas

Liste as notas fiscais com status `Autorizada`, mostrando:

- `nota_id`
- `data_emissao`
- `cliente_id`
- `uf_destino`
- `status_nota`

Ordene da emissão mais recente para a mais antiga.


In [55]:
query = """
SELECT
    nota_id,
    data_emissao,
    cliente_id,
    uf_destino,
    status_nota
FROM notas_fiscais
WHERE status_nota = 'Autorizada'
ORDER BY data_emissao DESC;
"""

con.execute(query).df()

,nota_id,data_emissao,cliente_id,uf_destino,status_nota
0,5007,2024-04-09,6,MG,Autorizada
1,5006,2024-03-22,5,RJ,Autorizada
2,5005,2024-03-05,1,SP,Autorizada
3,5004,2024-02-18,4,SP,Autorizada
4,5002,2024-01-20,2,SC,Autorizada
5,5001,2024-01-08,1,SP,Autorizada


## Exercício 9 — Notas com razão social do cliente

Use `INNER JOIN` para listar as notas fiscais com o nome do cliente.

Mostre:

- `nota_id`
- `data_emissao`
- `razao_social`
- `segmento`
- `uf_destino`
- `status_nota`

Ordene por `data_emissao`.


In [58]:
query = """
SELECT
    nf.nota_id,
    nf.data_emissao,
    c.razao_social,
    c.segmento,
    nf.uf_destino,
    nf.status_nota
FROM notas_fiscais nf
INNER JOIN clientes_fiscais c
    ON nf.cliente_id = c.cliente_id
ORDER BY nf.data_emissao;
"""

con.execute(query).df()

,nota_id,data_emissao,razao_social,segmento,uf_destino,status_nota
0,5001,2024-01-08,Alpha Ltda,Indústria,SP,Autorizada
1,5002,2024-01-20,Beta Comércio,Comércio,SC,Autorizada
2,5003,2024-02-10,Casa Norte,Comércio,PR,Cancelada
3,5004,2024-02-18,Delta Serviços,Serviços,SP,Autorizada
4,5005,2024-03-05,Alpha Ltda,Indústria,SP,Autorizada
5,5006,2024-03-22,Escola Saber,Educação,RJ,Autorizada
6,5007,2024-04-09,Farmácia Vida,Saúde,MG,Autorizada
7,5008,2024-04-25,Beta Comércio,Comércio,SC,Cancelada


## Exercício 10 — Valor total vendido por UF de destino

Calcule o valor total dos itens das notas por `uf_destino`.

Considere apenas notas com status `Autorizada`.

Mostre:

- UF de destino
- quantidade de notas distintas
- valor total dos itens
- valor total de imposto

Ordene pelo maior valor total dos itens.


In [70]:
query = """
SELECT
    nf.uf_destino AS UF_destino,
    COUNT(DISTINCT nf.nota_id) AS qtd_nf,
    SUM(itn.valor_unitario) AS valor_total_itens,
    SUM(itn.aliquota_imposto) AS valor_total_imposto
FROM notas_fiscais nf
INNER JOIN itens_notas itn
    ON nf.nota_id = itn.nota_id
WHERE nf.status_nota = 'Autorizada'
GROUP BY UF_destino
ORDER BY valor_total_itens DESC;        
"""

con.execute(query).df()

,UF_destino,qtd_nf,valor_total_itens,valor_total_imposto
0,SP,3,8450.0,0.32
1,SC,1,1200.0,0.12
2,MG,1,1150.0,0.12
3,RJ,1,850.0,0.05


In [60]:
print(con.execute('SELECT * FROM notas_fiscais LIMIT 1').df())
#print(con.execute('SELECT * FROM clientes_fiscais LIMIT 1').df())
print(con.execute('SELECT * FROM itens_notas LIMIT 1').df())
# alunos - aulas_assistidas - centros_custo - clientes_fiscais - contas_pagar - contas_receber
# cursos - itens_notas - matriculas - movimentacoes_bancarias - notas_fiscais - produtos_fiscais


   nota_id  cliente_id data_emissao status_nota uf_destino
0     5001           1   2024-01-08  Autorizada         SP
   item_id  nota_id  produto_id  quantidade  valor_unitario  aliquota_imposto  \
0        1     5001         201           2          3500.0              0.12   

   valor_item  valor_imposto  
0      7000.0          840.0  


## Exercício 11 — Receita fiscal por tipo: Produto ou Serviço

Use as tabelas `notas_fiscais`, `itens_notas` e `produtos_fiscais` para calcular o valor total por tipo.

Mostre:

- tipo
- quantidade de itens
- valor total
- imposto total

Considere apenas notas autorizadas.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 12 — Clientes com faturamento acima de R$ 5.000

Usando `GROUP BY` e `HAVING`, liste os clientes com valor total autorizado acima de R$ 5.000.

Mostre:

- razão social
- segmento
- valor total
- imposto total

Ordene pelo maior valor total.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 13 — Emissões por trimestre

Use `EXTRACT(QUARTER FROM data_emissao)` para contar quantas notas foram emitidas por trimestre.

Mostre:

- ano
- trimestre
- quantidade de notas

Ordene por ano e trimestre.


In [ ]:
query = """

"""

con.execute(query).df()

# Parte C — Financeiro

Nesta parte, você irá responder perguntas sobre contas a pagar, contas a receber, centros de custo e movimentações bancárias.


## Exercício 14 — Contas a pagar em aberto

Liste todas as contas a pagar com status `Aberto`.

Mostre:

- fornecedor
- data de vencimento
- valor
- status

Ordene pela data de vencimento.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 15 — Despesas por centro de custo

Use `INNER JOIN` entre `contas_pagar` e `centros_custo` para calcular o total de despesas por centro de custo.

Mostre:

- centro de custo
- quantidade de contas
- valor total
- valor médio

Ordene pelo maior valor total.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 16 — Recebimentos por cliente

Calcule o total de contas a receber por cliente.

Mostre:

- cliente
- quantidade de títulos
- valor total
- valor médio

Ordene pelo maior valor total.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 17 — Clientes com recebimentos acima de R$ 6.000

Usando `HAVING`, liste apenas clientes com valor total a receber acima de R$ 6.000.

Mostre:

- cliente
- valor total

Ordene do maior para o menor.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 18 — Fluxo financeiro mensal

Usando a tabela `movimentacoes_bancarias`, calcule entrada, saída e saldo por mês.

Dica: use `CASE WHEN` dentro do `SUM`.

Mostre:

- mês de referência
- total de entradas
- total de saídas
- saldo do mês

Ordene cronologicamente.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 19 — Atraso em contas a pagar

Liste as contas pagas depois do vencimento.

Mostre:

- fornecedor
- data de vencimento
- data de pagamento
- valor
- dias de atraso

Dica: no DuckDB, você pode subtrair datas para calcular diferença em dias.


In [ ]:
query = """

"""

con.execute(query).df()

## Exercício 20 — Análise financeira integrada

Monte uma consulta que responda:

**Qual foi o total de entradas, saídas e saldo acumulado considerando todas as movimentações bancárias?**

Mostre apenas uma linha com:

- total de entradas
- total de saídas
- saldo final


In [ ]:
query = """

"""

con.execute(query).df()

# Desafio final — Perguntas de negócio

Escolha **uma pergunta de negócio** abaixo e escreva uma consulta SQL para respondê-la.

1. Qual categoria de curso gera mais receita potencial?
2. Qual UF concentra maior valor autorizado em notas fiscais?
3. Qual centro de custo representa maior despesa?
4. Qual mês teve melhor saldo financeiro?
5. Quais clientes fiscais também aparecem nas contas a receber?

Escreva sua consulta abaixo.


In [ ]:
query = """

"""

con.execute(query).df()